<a href="https://colab.research.google.com/github/GustavoSant1/descoberta-cientifica2/blob/main/main0505.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Célula 1 - Instalações e Imports


In [7]:

!pip install requests beautifulsoup4 -q

import requests
from bs4 import BeautifulSoup
import sqlite3
import logging
import pandas as pd
from typing import List

# Configuração para mostrar as mensagens de forma limpa
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
print("✅ Célula 1 executada: Bibliotecas importadas com sucesso!")

✅ Célula 1 executada: Bibliotecas importadas com sucesso!


# Célula 2 - Lógica de Extração


In [8]:

class NewsScraper:
    def __init__(self, url: str):
        self.url = url
        self.headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

    def fetch_headlines(self) -> List[str]:
        try:
            response = requests.get(self.url, headers=self.headers, timeout=10)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, 'html.parser')
            headlines = []

            # Buscando as classes específicas do G1
            for item in soup.find_all('a', class_='feed-post-link'):
                text = item.get_text(strip=True)
                if text:
                    headlines.append(text)

            return headlines
        except Exception as e:
            logging.error(f"Erro ao acessar {self.url}: {e}")
            return []

print("✅ Célula 2 executada: Classe NewsScraper carregada na memória.")

✅ Célula 2 executada: Classe NewsScraper carregada na memória.


# Célula 3 - Lógica do Banco de Dados


In [10]:

class DatabaseManager:
    def __init__(self, db_name: str = "noticias.db"):
        self.db_name = db_name
        self._create_table()

    def _create_table(self):
        with sqlite3.connect(self.db_name) as conn:
            cursor = conn.cursor()
            cursor.execute('''
                CREATE TABLE IF NOT EXISTS manchetes (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    titulo TEXT NOT NULL UNIQUE
                )
            ''')
            conn.commit()

    def save_headlines(self, headlines: List[str]):
        with sqlite3.connect(self.db_name) as conn:
            cursor = conn.cursor()
            saved_count = 0
            for headline in headlines:
                try:
                    cursor.execute('INSERT INTO manchetes (titulo) VALUES (?)', (headline,))
                    saved_count += 1
                except sqlite3.IntegrityError:
                    pass
            conn.commit()
            logging.info(f"{saved_count} novas manchetes salvas no banco de dados!")

print("✅ Célula 3 executada: Classe DatabaseManager carregada na memória.")

✅ Célula 3 executada: Classe DatabaseManager carregada na memória.


# Célula 5 - Lendo o banco de dados e mostrando na tela


In [11]:
conn = sqlite3.connect("noticias.db")
df_manchetes = pd.read_sql_query("SELECT * FROM manchetes", conn)
conn.close()

print(f"Total de manchetes no banco de dados: {len(df_manchetes)}")
df_manchetes.head(5)

Total de manchetes no banco de dados: 9


,id,titulo
0,1,Lula quer afastar ideia de Trump de equiparar ...
1,2,"Dólar cai a R$ 4,91, menor valor desde janeiro..."
2,3,Jovem armado mata duas funcionárias e fere alu...
3,4,'A professora mandou a gente sentar perto da p...
4,5,Irmã enviou mensagem a piloto ao ver na notíci...
